In [1]:
print("Hello world")

Hello world


### API-Based Models

In [ ]:
from openai import OpenAI

client = OpenAI()

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role":"system", "content":"You are a helpful assistant"},
        {"role":"user", "content":"explain RAG briefly"}
    ],
    temparature=0.2
)

print(resp.choices[0].message.content)

### Local model inference(HuggingFace)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "meta-llama/Llama-2-7b-chat-hf"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

inputs = tok("Explain LoRA", return_tensor="pt")
outputs = model.generate(**inputs, max_new_tokens=100)
print(tok.decode(outputs[0]))

### Canonical RAG Flow

In [ ]:
# graphql

User Query
 → Query Rewriting
 → Retriever (BM25 + Vector)
 → Reranker
 →Prompt assembly
 →LLM

### Simple RAG(vector search)

In [ ]:
query_emb = embedder.encode(query)
results = vectordb.search(query_emb, top_k=5)

context = "\n".join([r.text for i in results])
prompt = f"Answer using context:\n{context}\n\nQuestion: {query}"

### Hybrid search(BM25 + Embeddings)

In [ ]:
bm25_hits = bm25.search(query)
vector_hits = vectordb.search(embed(query))

combined = merge_and_score(bm25_hits, vector_hits)

### Reranking (cross-encoder)

In [ ]:
from sentence_transformer import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
pairs = [(query, doc.text) for doc in candidates]
scores = reranker.predict(pairs)

## Vector DBs (pinecone, Weaviate, Qdrant)

### QDrant

In [ ]:
from qdrant_client import QdrantClient

client = QdrantClient(":memory:")

client.upsert(
    collection_names="docs",
    points=[
        {"id":1, "vector":embedding, "payload":{"source": "pdf"}}
    ]
)

hits = client.search("docs", query_vector=embedding, limit=5)